# 🏥 Multimodal AI Benchmark on ROP-VL Dataset
### Vision-Language & Multimodal Systems in Clinical Settings
**Course / Project:** FAST-NUCES Final Year Project (FYP)  
**Objective:** Build and benchmark a 3-modality pipeline on the **ROP-VL** dataset (Nature Scientific Data, August 2026) comparing **Early Fusion**, **Late Fusion**, and **Intermediate Cross-Attention / GAT Fusion**.

---
### Pipeline Overview:
1. **Vision Modality:** 2D Color Fundus Retinal Photography (CFPs) $\rightarrow$ Pretrained ResNet-50 / ViT
2. **Language Modality:** Expert clinical descriptions ($\pi_{\text{ROP-EK}}$ template) $\rightarrow$ Pretrained BioClinicalBERT
3. **Structured Tabular Modality:** Patient metadata (Gestational Age, Birth Weight, Gender, Plurality, Delivery Mode) $\rightarrow$ 3-Layer MLP
4. **Fusion Paradigms Evaluated:**
   - **Early Fusion:** Feature concatenation $[v_{\text{img}} \,;\, v_{\text{txt}} \,;\, v_{\text{tab}}]$
   - **Late Fusion:** Decision-level probability ensembling
   - **Intermediate Fusion:** Relational Cross-Attention message passing between modality graph nodes

## ⚙️ Step 1: Environment Setup & Dependencies

In [ ]:
!nvidia-smi
!pip install -q transformers timm openpyxl seaborn scikit-learn tabulate

## 📥 Step 2: Download & Extract ROP-VL Dataset
Downloads `ROP-VL.zip` (~656 MB) directly from the official Figshare repository (DOI: `10.6084/m9.figshare.30143461.v1`).

In [ ]:
import os
import zipfile
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
zip_path = os.path.join(DATA_DIR, "ROP-VL.zip")

download_url = "https://ndownloader.figshare.com/files/58329352"

if not os.path.exists(os.path.join(DATA_DIR, "img_info.xlsx")):
    if not os.path.exists(zip_path):
        print("[*] Downloading ROP-VL.zip (~656 MB) from Figshare...")
        urllib.request.urlretrieve(download_url, zip_path)
        print("[+] Download complete!")

    print("[*] Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("[+] Extraction complete!")
else:
    print("[+] ROP-VL dataset already extracted and ready in:", DATA_DIR)

!ls -lah data

## 📊 Step 3: Dataset Exploration & 3-Modality PyTorch DataLoader
Implements **Patient-Level 5-Fold Partitioning** on `patient_id` to strictly prevent data leakage across bilateral eyes and follow-up visits.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
from torchvision import transforms
from transformers import AutoTokenizer

# 4-Class Occurrence Mapping according to Nature Scientific Data paper
OCCURRENCE_4_CLASSES = {
    'Normal': 0,
    'Laser-treated ROP': 1,
    'Laser-treated': 1,
    'Stage 1 ROP': 2,
    'Stage 2 ROP': 2,
    'Stage 3 ROP': 2,
    'Stage 4 ROP': 2,
    'Stage 5 ROP': 2,
    'A-ROP': 3
}

class ROPVLDataset(Dataset):
    def __init__(self, data_dir, split="train", tokenizer_name="emilyalsentzer/Bio_ClinicalBERT", img_size=224, fold=0, total_folds=5):
        self.data_dir = data_dir
        self.split = split
        self.img_size = img_size

        img_info_path = os.path.join(data_dir, "img_info.xlsx")
        disease_desc_path = os.path.join(data_dir, "disease_description.xlsx")

        self.df = pd.read_excel(img_info_path)
        
        self.disease_desc_dict = {}
        if os.path.exists(disease_desc_path):
            desc_df = pd.read_excel(disease_desc_path)
            for _, row in desc_df.iterrows():
                cat = str(row.iloc[0]).strip()
                desc = str(row.iloc[1]).strip()
                self.disease_desc_dict[cat] = desc

        self.label_map = OCCURRENCE_4_CLASSES
        self.num_classes = 4

        # 5-Fold Patient-Level Partitioning
        unique_patients = np.sort(self.df['patient_id'].unique())
        np.random.seed(42)
        shuffled_patients = np.random.permutation(unique_patients)
        folds = np.array_split(shuffled_patients, total_folds)
        val_patients = set(folds[fold])

        if split == "train":
            self.df = self.df[~self.df['patient_id'].isin(val_patients)].reset_index(drop=True)
        else:
            self.df = self.df[self.df['patient_id'].isin(val_patients)].reset_index(drop=True)

        self.ga_mean, self.ga_std = 30.40, 3.85
        self.bw_mean, self.bw_std = 1490.37, 732.03

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

        if split == "train":
            self.img_transforms = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.img_transforms = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['image'])
        category = str(row['categories']).strip()
        
        possible_paths = [
            os.path.join(self.data_dir, category, img_name),
            os.path.join(self.data_dir, "images", category, img_name),
            os.path.join(self.data_dir, "images", img_name),
            os.path.join(self.data_dir, img_name)
        ]
        
        img_path = None
        for p in possible_paths:
            if os.path.exists(p):
                img_path = p
                break

        if img_path and os.path.exists(img_path):
            image = Image.open(img_path).convert('RGB')
        else:
            image = Image.new('RGB', (self.img_size, self.img_size), color=(120, 60, 40))

        img_tensor = self.img_transforms(image)

        # Language Template: pi_ROP-EK
        desc_text = self.disease_desc_dict.get(category, f"clinical signs of {category}")
        gender = str(row.get('gender', 'unknown'))
        ga = row.get('gestational_age', 30)
        bw = row.get('birth_weight', 1500)
        plurality = "singleton" if row.get('plurality', 1) == 1 else "multiple birth"
        dm = str(row.get('delivery_mode', 'cesarean section'))

        full_text = f"A fundus photograph of {desc_text} in a {gender} infant born at {ga} weeks with birth weight of {bw}g from {plurality} delivered via {dm}."
        tokenized = self.tokenizer(full_text, padding="max_length", truncation=True, max_length=96, return_tensors="pt")

        # Tabular Scaling
        gender_code = 1.0 if gender.lower() == 'male' else 0.0
        ga_norm = (float(ga) - self.ga_mean) / self.ga_std
        bw_norm = (float(bw) - self.bw_mean) / self.bw_std
        plurality_code = float(row.get('plurality', 1)) - 1.0
        dm_code = 1.0 if 'vag' in dm.lower() else 0.0
        tabular_vector = torch.tensor([gender_code, ga_norm, bw_norm, plurality_code, dm_code], dtype=torch.float32)

        label = self.label_map.get(category, 0)
        return {
            'image': img_tensor,
            'input_ids': tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'tabular': tabular_vector,
            'label': torch.tensor(label, dtype=torch.long),
            'text': full_text,
            'patient_id': row['patient_id']
        }

train_ds = ROPVLDataset(DATA_DIR, split="train")
val_ds = ROPVLDataset(DATA_DIR, split="val")
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"[*] Dataset Loaded: Train={len(train_ds)} | Val={len(val_ds)} | Classes={train_ds.num_classes}")

## 🧠 Step 4: Modality Encoders & Fusion Architectures
Defines:
1. **`VisionEncoder`:** ResNet-50 backbone $\rightarrow$ Linear $[2048 \rightarrow 256]$
2. **`TextEncoder`:** BioClinicalBERT backbone $\rightarrow$ Linear $[768 \rightarrow 256]$
3. **`TabularEncoder`:** 3-Layer MLP $[5 \rightarrow 64 \rightarrow 128 \rightarrow 256]$
4. **`EarlyFusionModel`:** Vector concatenation $[v_{\text{img}} \,;\, v_{\text{txt}} \,;\, v_{\text{tab}}]$
5. **`LateFusionModel`:** Independent heads + Learnable soft-voting
6. **`IntermediateFusionModel`:** Modality graph nodes + Cross-Attention message passing

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel

class VisionEncoder(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projector = nn.Sequential(
            nn.Linear(in_features, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, images):
        features = self.backbone(images)
        return self.projector(features)

class TextEncoder(nn.Module):
    def __init__(self, model_name="emilyalsentzer/Bio_ClinicalBERT", embed_dim=256, freeze_bert=True):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained(model_name)
        except Exception:
            self.bert = AutoModel.from_pretrained("bert-base-uncased")

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        hidden_size = self.bert.config.hidden_size
        self.projector = nn.Sequential(
            nn.Linear(hidden_size, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.projector(cls_token)

class TabularEncoder(nn.Module):
    def __init__(self, in_features=5, embed_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, tabular_data):
        return self.mlp(tabular_data)

# 1. Early Fusion
class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256):
        super().__init__()
        self.vision_encoder = VisionEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(embed_dim=embed_dim)
        self.tabular_encoder = TabularEncoder(in_features=5, embed_dim=embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, batch):
        v = self.vision_encoder(batch['image'])
        t = self.text_encoder(batch['input_ids'], batch['attention_mask'])
        s = self.tabular_encoder(batch['tabular'])
        fused = torch.cat([v, t, s], dim=-1)
        return self.classifier(fused), {}

# 2. Late Fusion
class LateFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256):
        super().__init__()
        self.vision_encoder = VisionEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(embed_dim=embed_dim)
        self.tabular_encoder = TabularEncoder(in_features=5, embed_dim=embed_dim)
        self.vision_head = nn.Linear(embed_dim, num_classes)
        self.text_head = nn.Linear(embed_dim, num_classes)
        self.tabular_head = nn.Linear(embed_dim, num_classes)
        self.modality_weights = nn.Parameter(torch.ones(3) / 3.0)

    def forward(self, batch):
        v_logits = self.vision_head(self.vision_encoder(batch['image']))
        t_logits = self.text_head(self.text_encoder(batch['input_ids'], batch['attention_mask']))
        s_logits = self.tabular_head(self.tabular_encoder(batch['tabular']))
        weights = torch.softmax(self.modality_weights, dim=0)
        final_logits = weights[0] * v_logits + weights[1] * t_logits + weights[2] * s_logits
        return final_logits, {'weights': weights.detach().cpu().numpy()}

# 3. Intermediate Fusion (Proposed Cross-Attention / GAT)
class IntermediateFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256, num_heads=4):
        super().__init__()
        self.vision_encoder = VisionEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(embed_dim=embed_dim)
        self.tabular_encoder = TabularEncoder(in_features=5, embed_dim=embed_dim)
        self.modality_embed = nn.Embedding(3, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, batch):
        B = batch['image'].size(0)
        device = batch['image'].device
        v = self.vision_encoder(batch['image'])
        t = self.text_encoder(batch['input_ids'], batch['attention_mask'])
        s = self.tabular_encoder(batch['tabular'])
        tokens = torch.stack([v, t, s], dim=1)
        mod_ids = torch.tensor([0, 1, 2], device=device).unsqueeze(0).expand(B, -1)
        tokens = tokens + self.modality_embed(mod_ids)
        attn_out, attn_weights = self.attn(query=tokens, key=tokens, value=tokens)
        tokens = self.norm(tokens + attn_out)
        fused = tokens.reshape(B, -1)
        return self.classifier(fused), {'attention_matrix': attn_weights.detach().cpu()}

print("[+] All 3 fusion architectures instantiated successfully!")

## 🏋️ Step 5: Training Helper Function

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, train_loader, val_loader, epochs=8, lr=1e-4, model_name="model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_f1 = 0.0

    print(f"\n---> Training {model_name.upper()} on {device} for {epochs} epochs <--- ")
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, tr_preds, tr_labels = 0.0, [], []
        for batch in train_loader:
            for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                batch[k] = batch[k].to(device)
            optimizer.zero_grad()
            logits, _ = model(batch)
            loss = criterion(logits, batch['label'])
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * batch['label'].size(0)
            tr_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            tr_labels.extend(batch['label'].cpu().numpy())

        scheduler.step()

        # Validation
        model.eval()
        val_loss, val_preds, val_labels, val_probs = 0.0, [], [], []
        with torch.no_grad():
            for batch in val_loader:
                for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                    batch[k] = batch[k].to(device)
                logits, _ = model(batch)
                loss = criterion(logits, batch['label'])
                val_loss += loss.item() * batch['label'].size(0)
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
                val_probs.extend(probs)
                val_preds.extend(np.argmax(probs, axis=-1))
                val_labels.extend(batch['label'].cpu().numpy())

        tr_acc = accuracy_score(tr_labels, tr_preds)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Acc: {tr_acc*100:.1f}% || Val Acc: {val_acc*100:.1f}% | Val Macro F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"best_{model_name}.pth")
            
    return model

## 🚀 Step 6: Train All 3 Multimodal Fusion Models

In [ ]:
# 1. Early Fusion
early_model = EarlyFusionModel(num_classes=4)
early_model = train_model(early_model, train_loader, val_loader, epochs=8, model_name="early")

# 2. Late Fusion
late_model = LateFusionModel(num_classes=4)
late_model = train_model(late_model, train_loader, val_loader, epochs=8, model_name="late")

# 3. Intermediate Cross-Attention / GAT Fusion
inter_model = IntermediateFusionModel(num_classes=4)
inter_model = train_model(inter_model, train_loader, val_loader, epochs=8, model_name="intermediate")

## 📈 Step 7: Comparative Benchmark & Evaluation Table
Compares Accuracy, Macro F1, and Macro AUC across the 3 fusion strategies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

def evaluate_benchmark(models_dict, val_loader):
    results = []
    for name, model in models_dict.items():
        model.eval()
        all_preds, all_labels, all_probs = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                    batch[k] = batch[k].to(device)
                logits, _ = model(batch)
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
                all_probs.extend(probs)
                all_preds.extend(np.argmax(probs, axis=-1))
                all_labels.extend(batch['label'].cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        try:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
        except Exception:
            auc = 0.0
        results.append({
            "Fusion Paradigm": name,
            "Accuracy (%)": round(acc * 100, 2),
            "Macro F1": round(f1, 4),
            "Macro AUC": round(auc, 4)
        })
    return pd.DataFrame(results)

models = {
    "Early Fusion (Concat)": early_model,
    "Late Fusion (Decision)": late_model,
    "Intermediate (Proposed GAT/Cross-Attn)": inter_model
}

res_df = evaluate_benchmark(models, val_loader)
print("\n================ MULTIMODAL FUSION BENCHMARK ================")
print(tabulate(res_df, headers='keys', tablefmt='psql', showindex=False))

# Plotting Comparison Bar Chart
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
sns.barplot(x="Fusion Paradigm", y="Accuracy (%)", data=res_df, palette="viridis")
plt.xticks(rotation=25, ha="right")
plt.title("Accuracy (%)")

plt.subplot(1, 3, 2)
sns.barplot(x="Fusion Paradigm", y="Macro F1", data=res_df, palette="viridis")
plt.xticks(rotation=25, ha="right")
plt.title("Macro F1 Score")

plt.subplot(1, 3, 3)
sns.barplot(x="Fusion Paradigm", y="Macro AUC", data=res_df, palette="viridis")
plt.xticks(rotation=25, ha="right")
plt.title("Macro AUC")

plt.tight_layout()
plt.show()

## 🔬 Step 8: Single-Patient Inference & Cross-Modal Attention Heatmap
Demonstrates clinical explainability by visualizing which modality contributed most to the final diagnosis.

In [ ]:
# Take a validation sample
sample_batch = next(iter(val_loader))
sample_input = {k: v[:1].to(device) for k, v in sample_batch.items() if k in ['image', 'input_ids', 'attention_mask', 'tabular']}

inter_model.eval()
with torch.no_grad():
    logits, aux = inter_model(sample_input)
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    attn_matrix = aux['attention_matrix'].squeeze(0).mean(dim=0).numpy() # Shape [3, 3]

classes = ['Normal', 'Laser-treated ROP', 'Any Stage of ROP', 'A-ROP']
pred_idx = np.argmax(probs)

print(f"\n[Patient Clinical Prediction]")
print(f"Predicted Diagnosis: {classes[pred_idx]} (Confidence: {probs[pred_idx]*100:.2f}%)")

# Plot Modality Attention Graph Matrix
modality_labels = ['Vision (Fundus)', 'Language (Notes)', 'Tabular (Metadata)']
plt.figure(figsize=(6, 5))
sns.heatmap(attn_matrix, annot=True, fmt=".2f", cmap="Blues", xticklabels=modality_labels, yticklabels=modality_labels)
plt.title("Cross-Modal Attention Matrix (Explainability Signal)", weight="bold")
plt.xlabel("Target Modality (Key)")
plt.ylabel("Source Modality (Query)")
plt.tight_layout()
plt.show()